# NLP2Shell — Phase 2: Fine-tuning (Qwen-0.5 + LoRA)

**Hardware:** Kaggle T4 GPU (16GB VRAM)  
**Base Model:** `Qwen-0.5` (0.5B)  
**Method:** LoRA via PEFT + SFTTrainer  
**Dataset:** nl2bash-custom (Hugging Face) — loaded directly, no file upload needed  

Run cells top to bottom. Do not skip any cell.

In [25]:
# Cell 1: Install dependencies
#import os
#os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # ← set before ANYTHING else

#!pip install -q --force-reinstall --no-deps "transformers==4.46.0"
#!pip install -q \
#    "peft>=0.10.0" \
#    "trl>=0.8.6" \
#    "bitsandbytes>=0.43.0" \
#    "accelerate>=0.29.0" \
#    "datasets>=2.18.0"

#print("Install complete. Restarting kernel...")

# Auto-restart so the new transformers version is actually loaded
#import IPython
#IPython.Application.instance().kernel.do_shutdown(True)

In [26]:
# Cell 2: Verify GPU and library versions
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # ← repeat here since kernel restarted

import torch
import transformers
import peft
import trl
import bitsandbytes as bnb

print(f"PyTorch      : {torch.__version__}")
print(f"Transformers : {transformers.__version__}")   # must show 4.46.x
print(f"PEFT         : {peft.__version__}")
print(f"TRL          : {trl.__version__}")
print(f"BitsAndBytes : {bnb.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU count     : {torch.cuda.device_count()}")  # must print 1 ← key check

if torch.cuda.is_available():
    print(f"GPU  : {torch.cuda.get_device_name(0)}")
    print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU found.")

assert torch.cuda.device_count() == 1, \
    f"Still seeing {torch.cuda.device_count()} GPUs — CUDA_VISIBLE_DEVICES not applied!"
assert "4.46" in transformers.__version__, \
    f"Wrong transformers version: {transformers.__version__} — re-run Cell 1 and restart kernel"

PyTorch      : 2.10.0+cu128
Transformers : 4.46.0
PEFT         : 0.18.1
TRL          : 0.17.0
BitsAndBytes : 0.49.2
CUDA available: True
GPU count     : 1
GPU  : Tesla T4
VRAM : 15.6 GB


In [27]:
# Cell 3: Load dataset from Kaggle input
from datasets import load_dataset

dataset_path      = "/kaggle/input/datasets/hassannawaz1423/nlp2shell-processed/train.jsonl"
eval_dataset_path = "/kaggle/input/datasets/hassannawaz1423/nlp2shell-processed/val.jsonl"

train_dataset = load_dataset("json", data_files=dataset_path, split="train")
val_dataset   = load_dataset("json", data_files=eval_dataset_path, split="train")

print(f"Train size : {len(train_dataset)}")
print(f"Val size   : {len(val_dataset)}")
print(f"Columns    : {train_dataset.column_names}")
print("\nSample row:")
print(train_dataset[0])

Train size : 26436
Val size   : 3304
Columns    : ['instruction', 'input', 'output']

Sample row:
{'instruction': 'Compress all files under current directory tree with gzip', 'input': '', 'output': 'find . -type f -print0 | xargs -0r gzip'}


In [28]:
# Cell 4: Confirm 'instruction' and 'output' columns exist
assert 'instruction' in train_dataset.column_names, \
    f"Missing 'instruction' column. Found: {train_dataset.column_names}"
assert 'output' in train_dataset.column_names, \
    f"Missing 'output' column. Found: {train_dataset.column_names}"

print("Column check passed. Dataset is ready.")

Column check passed. Dataset is ready.


In [29]:
# Cell 6: Load tokenizer
from transformers import AutoTokenizer

model_id = "Qwen/Qwen2.5-0.5B-Instruct"   # ← only change

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Vocab size   : {tokenizer.vocab_size}")
print(f"Pad token    : {tokenizer.pad_token}")
print(f"Padding side : {tokenizer.padding_side}")

Vocab size   : 151643
Pad token    : <|im_end|>
Padding side : right


In [30]:
# Cell 7: Load model in 4-bit quantization
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map={"": 0},
    torch_dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)
model.config.use_cache = False

print("Model loaded.")
print(f"Device : {next(model.parameters()).device}")   # must show cuda:0
print(f"Dtype  : {next(model.parameters()).dtype}")

Model loaded.
Device : cuda:0
Dtype  : torch.float32


In [31]:
# Cell 8: Apply LoRA adapter
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


In [32]:
# Cell 9: Format dataset using Qwen chat template
# ROOT CAUSE FIX: formatting_func was training on full text without
# masking the instruction — model memorized prompts → loss=0, val=nan
# Chat template handles this correctly automatically

def format_to_chat(example):
    """Convert to Qwen chat format so SFTTrainer masks prompt tokens correctly."""
    return {
        "text": tokenizer.apply_chat_template(
            [
                {"role": "system",  "content": "Convert the natural language instruction to a bash command. Output only the command, nothing else."},
                {"role": "user",    "content": example["instruction"]},
                {"role": "assistant","content": example["output"]},
            ],
            tokenize=False,
            add_generation_prompt=False,
        )
    }

train_dataset_fmt = train_dataset.map(format_to_chat, remove_columns=train_dataset.column_names)
val_dataset_fmt   = val_dataset.map(format_to_chat,   remove_columns=val_dataset.column_names)

print("Sample formatted text:")
print(train_dataset_fmt[0]["text"])

Sample formatted text:
<|im_start|>system
Convert the natural language instruction to a bash command. Output only the command, nothing else.<|im_end|>
<|im_start|>user
Compress all files under current directory tree with gzip<|im_end|>
<|im_start|>assistant
find . -type f -print0 | xargs -0r gzip<|im_end|>



In [33]:
# Cell 10: Training arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/kaggle/working/nlp2shell-qwen-lora",
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    num_train_epochs=1,
    warmup_steps=200,
    logging_steps=50,
    save_strategy="steps",
    save_steps=300,
    save_total_limit=3,
    eval_strategy="steps",
    eval_steps=300,
    fp16=False,
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    dataloader_pin_memory=False,
    optim="paged_adamw_8bit",
    max_grad_norm=0.3,
    lr_scheduler_type="cosine",
    push_to_hub=False,
    report_to="none",
    load_best_model_at_end=False,
)

In [34]:
# Cell 11: Build SFTTrainer
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset_fmt,
    eval_dataset=val_dataset_fmt,
    processing_class=tokenizer,
    args=training_args,
)

print("Trainer built. Ready.")

Trainer built. Ready.


In [35]:
# Cell 11.5: Pre-training validation
import torch

print("=" * 50)
print("PRE-TRAINING CHECKS")
print("=" * 50)

# Check 1: Model device
devices = set(str(p.device) for p in model.parameters())
print(f"\n✓ Devices: {devices}")
assert all("cuda" in d for d in devices), "❌ Model not on GPU!"
print("  GPU — OK")

# Check 2: Trainable params
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"\n✓ Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")
assert 0 < trainable < total, "❌ LoRA not applied!"
print("  LoRA — OK")

# Check 3: Dataset has text column
assert "text" in train_dataset_fmt.column_names, "❌ Missing 'text' column!"
sample_text = train_dataset_fmt[0]["text"]
print(f"\n✓ Sample text:\n{sample_text[:300]}")
assert "<|im_start|>" in sample_text, "❌ Chat template not applied!"
assert "bash" in sample_text.lower() or "find" in sample_text.lower() \
    or "mv" in sample_text.lower() or "ls" in sample_text.lower(), \
    "❌ Output command missing from sample!"
print("  Format — OK")

# Check 4: Token length
tokens = tokenizer(sample_text, return_tensors="pt")
tok_len = tokens["input_ids"].shape[1]
print(f"\n✓ Token length: {tok_len}")
assert tok_len < 512, f"❌ Too long: {tok_len}"
print("  Length — OK")

# Check 5: Forward pass with real loss
print("\n✓ Forward pass...")
model.eval()
mini_texts  = [train_dataset_fmt[i]["text"] for i in range(4)]
mini_inputs = tokenizer(
    mini_texts,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=256,
).to("cuda:0")

with torch.no_grad():
    out  = model(**mini_inputs, labels=mini_inputs["input_ids"])
    loss = out.loss.item()

print(f"  Loss: {loss:.4f}")
assert not torch.isnan(torch.tensor(loss)), "❌ NaN loss!"
assert 0.5 < loss < 12.0, f"❌ Loss out of expected range: {loss:.4f}"
print("  Loss range — OK")

# Check 6: Backward pass
print("\n✓ Backward pass...")
model.train()
out2 = model(**mini_inputs, labels=mini_inputs["input_ids"])
out2.loss.backward()
grads = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
assert len(grads) > 0, "❌ No gradients!"
model.zero_grad()
model.train()
print(f"  Gradients on {len(grads)} layers — OK")

print("\n" + "=" * 50)
print("ALL CHECKS PASSED — run Cell 12")
print("=" * 50)

PRE-TRAINING CHECKS

✓ Devices: {'cuda:0'}
  GPU — OK

✓ Trainable: 8,798,208 / 323,917,696 (2.72%)
  LoRA — OK

✓ Sample text:
<|im_start|>system
Convert the natural language instruction to a bash command. Output only the command, nothing else.<|im_end|>
<|im_start|>user
Compress all files under current directory tree with gzip<|im_end|>
<|im_start|>assistant
find . -type f -print0 | xargs -0r gzip<|im_end|>

  Format — OK

✓ Token length: 58
  Length — OK

✓ Forward pass...
  Loss: 4.6186
  Loss range — OK

✓ Backward pass...


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


  Gradients on 336 layers — OK

ALL CHECKS PASSED — run Cell 12


In [36]:
# Cell 11.9: Fix PyTorch 2.6 checkpoint RNG loading bug
import torch
import numpy as np

torch.serialization.add_safe_globals([
    np.ndarray,
    np._core.multiarray._reconstruct,
    np.dtype,
    np.dtypes.UInt32DType,
])

print("PyTorch 2.6 RNG globals patched.")

PyTorch 2.6 RNG globals patched.


In [37]:
# Cell 12: TRAIN
import os

checkpoint_dir = "/kaggle/working/nlp2shell-qwen-lora"
checkpoints = [
    f for f in os.listdir(checkpoint_dir)
    if f.startswith("checkpoint")
] if os.path.exists(checkpoint_dir) else []

if checkpoints:
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
    resume = os.path.join(checkpoint_dir, latest)
    print(f"Resuming from: {resume}")
else:
    resume = None
    print("Starting fresh.")

print("Starting training...")
trainer.train(resume_from_checkpoint=resume)
print("Training complete.")

Resuming from: /kaggle/working/nlp2shell-qwen-lora/checkpoint-250
Starting training...


Step,Training Loss,Validation Loss
300,0.962200,0.983373
350,0.960900,0.962393
400,0.965200,0.944953
450,0.946500,0.934453
500,0.915100,0.920834
550,0.918200,0.905860
600,0.922900,0.897757
650,0.883700,0.885998
700,0.865200,0.877743
750,0.938000,0.871685


Training complete.


In [38]:
# Cell 13: Save adapter
SAVE_PATH = "/kaggle/working/qwen_final_adapter"

trainer.model.save_pretrained(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

print(f"Adapter saved to: {SAVE_PATH}")

import os
print(f"Saved files: {os.listdir(SAVE_PATH)}")

Adapter saved to: /kaggle/working/qwen_final_adapter
Saved files: ['README.md', 'added_tokens.json', 'vocab.json', 'tokenizer.json', 'special_tokens_map.json', 'adapter_config.json', 'tokenizer_config.json', 'merges.txt', 'adapter_model.safetensors']


In [44]:
# Cell 14: Smoke test
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

SAVE_PATH = "/kaggle/working/qwen_final_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

inf_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-0.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)
inf_model = PeftModel.from_pretrained(inf_model, SAVE_PATH)
inf_model.eval()
inf_tokenizer = AutoTokenizer.from_pretrained(SAVE_PATH)

test_inputs = [
    "move all PDF files from Downloads to Documents",
    "list all files in the current directory including hidden ones",
    "find all files larger than 100MB",
    "create a new directory called projects inside home",
    "show the first 20 lines of a file called log.txt",
    "kill the process running on port 8080",
    "show disk usage of each folder in current directory",
    "compress the folder named backup into a tar.gz file",
]

SYSTEM_PROMPT = "Convert the natural language instruction to a bash command. Output only the command, nothing else."

print("=" * 60)
print("SMOKE TEST — predictions")
print("=" * 60)

for nl in test_inputs:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": nl},
    ]
    prompt = inf_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = inf_tokenizer(prompt, return_tensors="pt").to(inf_model.device)

    with torch.no_grad():
        outputs = inf_model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=80,
            do_sample=False,
            eos_token_id=inf_tokenizer.eos_token_id,
            pad_token_id=inf_tokenizer.eos_token_id,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    predicted = inf_tokenizer.decode(generated, skip_special_tokens=True).strip()

    print(f"\nNL     : {nl}")
    print(f"Command: {predicted}")
    print("-" * 60)

print("\nSmoke test complete.")

SMOKE TEST — predictions

NL     : move all PDF files from Downloads to Documents
Command: mv ~/Downloads/*.pdf ~/Documents/
------------------------------------------------------------

NL     : list all files in the current directory including hidden ones
Command: find . -type f -print0 | xargs -0 ls -l
------------------------------------------------------------

NL     : find all files larger than 100MB
Command: find / -size +100M
------------------------------------------------------------

NL     : create a new directory called projects inside home
Command: mkdir ~/projects
------------------------------------------------------------

NL     : show the first 20 lines of a file called log.txt
Command: cat -n log.txt | head -n 20
------------------------------------------------------------

NL     : kill the process running on port 8080
Command: kill -9 $(netstat -tuln | grep '8080' | awk '{print $2}')
------------------------------------------------------------

NL     : show disk